In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/automatizacion/

/content/drive/MyDrive/automatizacion/Aut4


In [ ]:
import os
import pandas as pd
import pathlib as Path

In [ ]:
paths = {'F1': '/content/drive/MyDrive/automatizacion/',
          'F2': '/content/drive/MyDrive/automatizacion/',
          'F3':'/content/drive/MyDrive/automatizacion/',
          'F4': '/content/drive/MyDrive/automatizacion/'}

clientes = ['F1', 'F2', 'F3', 'F4']

In [ ]:
def ultimo_generado(farmacia):
  """
  Busca en la carpeta de la farmacia, filtra solo los Excel,
  los ordena por fecha real de modificación y devuelve la ruta completa del más reciente.
  """
  directorio = paths[farmacia]

  # Listar todo lo que hay en el directorio
  archivos = os.listdir(directorio)

  # Filtrar para asegurarnos de tomar solo archivos .xlsx (evita errores con carpetas o archivos ocultos)
  archivos_excel = [f for f in archivos if f.endswith('.xlsx')]

  if not archivos_excel:
    raise FileNotFoundError(f"No se encontraron archivos Excel en {directorio}")

  # Armar las rutas completas (Directorio + Nombre del archivo)
  rutas_completas = [os.path.join(directorio, f) for f in archivos_excel]

  # Ordenar la lista basándonos en el "Modification Time" (mtime) del sistema operativo
  rutas_completas.sort(key=os.path.getmtime)

  # El último elemento de la lista ordenada es el más reciente
  ruta_mas_reciente = rutas_completas[-1]

  return ruta_mas_reciente

In [ ]:
# 2. Cargar los 4 Excels y extraer solo lo necesario (Columnas B, C y E)
dfs = {}

for cliente in clientes:
  try:
    archivo_a_cargar = ultimo_generado(cliente)
    # Solo para que veas en consola qué archivo exacto está tomando
    print(f"[{cliente}] Cargando último archivo: {os.path.basename(archivo_a_cargar)}")

    # Leemos el excel pasando la ruta absoluta
    df = pd.read_excel(archivo_a_cargar, usecols="A,C,E")

    # Renombramos "Precio Final" para saber de quién es
    df.rename(columns={'Precio Final': f'Precio_{cliente}'}, inplace=True)
    dfs[cliente] = df

  except Exception as e:
    print(f"Error cargando los datos de {cliente}: {e}")

In [ ]:
from functools import reduce

# --- 3. EL GRAN CRUCE (MERGE) ---
# Extraemos solo los DataFrames que se cargaron exitosamente
lista_dfs = list(dfs.values())

In [ ]:
# Usamos 'reduce' para ir pegando las tablas una por una.
# Usamos how='outer' para no perder ningún producto, incluso si alguna farmacia no lo vende.
df_maestro = reduce(
    lambda izquierda, derecha: pd.merge(izquierda, derecha, on=['CODIGO BARRA', 'PRODUCTO'], how='outer'),
    lista_dfs
)

In [ ]:
# --- 4. CÁLCULO DE COMPETITIVIDAD ---
# Creamos una lista dinámica con los nombres de las columnas de precio
columnas_precios = [f'Precio_{c}' for c in dfs.keys()]
columnas_precios

In [ ]:
# Calculamos el precio más bajo del mercado por cada producto (ignora automáticamente los vacíos/NaN)
df_maestro['Precio_Minimo'] = df_maestro[columnas_precios].min(axis=1)
df_maestro

In [ ]:
# Encontramos QUÉ columna tiene ese precio mínimo (idxmin) y le quitamos el prefijo "Precio_" para dejar solo el nombre
df_maestro['Lider_Actual'] = df_maestro[columnas_precios].idxmin(axis=1).str.replace('Precio_', '')
df_maestro

In [ ]:
# Vemos un pequeño resumen en consola para confirmar que funcionó
print(f"Total de productos únicos en el mercado: {len(df_maestro)}")
print(df_maestro[['CODIGO BARRA', 'PRODUCTO', 'Precio_Minimo', 'Lider_Actual']].head())

In [ ]:
# --- 5. GENERACIÓN DE ALERTAS (EL FILTRO POR F1) ---
# Filtramos donde FV no es lider
# Asegurarnos de que el cliente sí vende el producto (que su precio no sea nulo/NaN).
condicion_alerta = (df_maestro['Lider_Actual'] != 'F1') & (df_maestro['Precio_F1'].notna())
condicion_alerta

,0
0,False
1,False
2,False
3,False
4,False
...,...
3916,False
3917,False
3918,False
3919,False


In [ ]:
df_alertas = df_maestro[condicion_alerta].copy()
df_alertas

In [ ]:
# Si el dataframe está vacío, significa que el cliente es el rey absoluto del mercado
if df_alertas.empty:
  print(f"[{cliente}] 🏆 ¡Excelente! Tienen el mejor precio en todo su catálogo.")
  continue # Saltamos al siguiente cliente

SyntaxError: 'continue' not properly in loop (545086303.py, line 4)

In [ ]:
# --- CÁLCULO DE LA BRECHA ---
# Fórmula: ((Mi_Precio - Precio_Competencia) / Precio_Competencia) * 100
df_alertas['Brecha_Porcentual'] = ((df_alertas['Precio_F1'] - df_alertas['Precio_Minimo']) / df_alertas['Precio_Minimo']) * 100
df_alertas

In [ ]:
# Redondeamos a 2 decimales para que el JSON quede limpio
df_alertas['Brecha_Porcentual'] = df_alertas['Brecha_Porcentual'].round(2)

In [ ]:
# --- PRIORIZACIÓN ---
# Ordenamos de mayor a menor brecha. Así, los casos más graves quedan de primeros en el JSON.
df_alertas = df_alertas.sort_values(by='Brecha_Porcentual', ascending=False)
df_alertas

In [ ]:
# --- LIMPIEZA FINAL ---
# Nos quedamos solo con las columnas que la IA necesita leer
columnas_finales = ['CODIGO BARRA', 'PRODUCTO', 'Precio_F1', 'Precio_Minimo', 'Lider_Actual', 'Brecha_Porcentual']
df_exportar = df_alertas[columnas_finales]
df_exportar

In [ ]:
# --- EXPORTACIÓN ---
# Limpiamos el nombre para el archivo
nombre_archivo = f"alertas_{'F1'.replace(' ', '').lower()}.json"
nombre_archivo

In [ ]:
# Guardamos el JSON. 'orient="records"' es vital para que n8n lo lea correctamente como una lista de items.
df_exportar.to_json(nombre_archivo, orient='records', force_ascii=False, indent=4)

print(f"[{cliente}] 🚨 Se generaron {len(df_exportar)} alertas de sobreprecio. Archivo: {nombre_archivo}")

In [ ]:
# --- 5. GENERACIÓN DE ALERTAS (EL FILTRO POR CLIENTE) ---

# Recorremos solo los clientes que logramos cargar exitosamente
for cliente in dfs.keys():
  mi_precio_col = f'Precio_{cliente}'

  # REGLA 1: Filtrar donde el cliente NO es el líder.
  # REGLA 2: Asegurarnos de que el cliente sí vende el producto (que su precio no sea nulo/NaN).
  condicion_alerta = (df_maestro['Lider_Actual'] != cliente) & (df_maestro[mi_precio_col].notna())
  df_alertas = df_maestro[condicion_alerta].copy()

  # Si el dataframe está vacío, significa que el cliente es el rey absoluto del mercado
  if df_alertas.empty:
    print(f"[{cliente}] 🏆 ¡Excelente! Tienen el mejor precio en todo su catálogo.")
    continue # Saltamos al siguiente cliente

  # --- CÁLCULO DE LA BRECHA ---
  # Fórmula: ((Mi_Precio - Precio_Competencia) / Precio_Competencia) * 100
  df_alertas['Brecha_Porcentual'] = ((df_alertas[mi_precio_col] - df_alertas['Precio_Minimo']) / df_alertas['Precio_Minimo']) * 100

  # Redondeamos a 2 decimales para que el JSON quede limpio
  df_alertas['Brecha_Porcentual'] = df_alertas['Brecha_Porcentual'].round(2)

  # --- PRIORIZACIÓN ---
  # Ordenamos de mayor a menor brecha. Así, los casos más graves quedan de primeros en el JSON.
  df_alertas = df_alertas.sort_values(by='Brecha_Porcentual', ascending=False)

  # --- LIMPIEZA FINAL ---
  # Nos quedamos solo con las columnas que la IA necesita leer
  columnas_finales = ['CODIGO BARRA', 'PRODUCTO', mi_precio_col, 'Precio_Minimo', 'Lider_Actual', 'Brecha_Porcentual']
  df_exportar = df_alertas[columnas_finales]

  # --- EXPORTACIÓN ---
  # Limpiamos el nombre para el archivo
  nombre_archivo = f"alertas_{cliente.replace(' ', '').lower()}.json"

  # Guardamos el JSON. 'orient="records"' es vital para que n8n lo lea correctamente como una lista de items.
  df_exportar.to_json(nombre_archivo, orient='records', force_ascii=False, indent=4)

  print(f"[{cliente}] 🚨 Se generaron {len(df_exportar)} alertas de sobreprecio. Archivo: {nombre_archivo}")